In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import torch
from transformers import AutoTokenizer
from src.eval.beaver import AutoModelForScore

model_name = "PKU-Alignment/beaver-7b-v1.0-cost"

model = AutoModelForScore.from_pretrained(
    model_name,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
    max_model_length=2048,
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import torch
import torch_tensorrt

model = torch.compile(model, backend="tensorrt")

In [ ]:
BEAVER_PROMPT = """BEGINNING OF CONVERSATION: USER: {behavior} ASSISTANT: {generation}"""

data = [
    ("I like to play football.", "I enjoy playing soccer with my friends."),
    ("I love reading books.", "I am fond of novels and literature."),
]

texts = [BEAVER_PROMPT.format(behavior=pair[0], generation=pair[1]) for pair in data]

In [ ]:
input_ids = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
).to(model.device)

with torch.inference_mode():
    outputs = model(**input_ids)
    
scores = outputs.end_scores
print(scores)

In [ ]:
input_ids = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
).to(model.device)

with torch.inference_mode():
    outputs = model(**input_ids)
    
scores = outputs.end_scores
print(scores)